# nb04_text_mining
**Chủ:** D  
Chủ đề phàn nàn trong review Negative; ghi mart.review_topic; precision + Wilson


In [1]:
import sys; sys.path.insert(0, '..')
from config import *


In [2]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(DB_URL)
sql = """
SELECT r.review_id, c.category, r.sentiment, s.review_text
FROM dw.fact_review r
JOIN dw.dim_category c USING (category_key)
JOIN stg.review_all_rows s ON s.src_row_id = r.src_row_id
WHERE r.sentiment = 'Negative'
"""
df = pd.read_sql(sql, engine)
print(df.shape)
print(df["category"].value_counts().head(10))

(6057, 4)
category
GAME                1430
FAMILY               667
FINANCE              300
SPORTS               284
TRAVEL_AND_LOCAL     267
TOOLS                239
DATING               228
PRODUCTIVITY         208
SOCIAL               190
PHOTOGRAPHY          184
Name: count, dtype: int64


In [3]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

sw = list(stopwords.words("english"))
vec = TfidfVectorizer(lowercase=True, stop_words=sw,
                      token_pattern=r"\b[a-zA-Z]{3,}\b",
                      min_df=5, max_df=0.5)
X = vec.fit_transform(df["review_text"].fillna(""))
scores = np.asarray(X.mean(axis=0)).ravel()
top = pd.Series(scores, index=vec.get_feature_names_out()).sort_values(ascending=False)
print(X.shape)
print(top.head(60).round(4).to_string())

(6057, 2471)
game        0.0553
time        0.0230
bad         0.0225
get         0.0210
like        0.0204
app         0.0177
ads         0.0174
play        0.0168
even        0.0164
hate        0.0153
update      0.0145
phone       0.0136
worst       0.0133
really      0.0131
please      0.0129
work        0.0125
annoying    0.0122
useless     0.0118
every       0.0116
fix         0.0116
waste       0.0116
fun         0.0113
would       0.0112
good        0.0110
slow        0.0102
money       0.0102
much        0.0100
want        0.0099
new         0.0098
ever        0.0096
back        0.0096
make        0.0096
sucks       0.0094
still       0.0094
need        0.0091
screen      0.0089
way         0.0088
also        0.0087
open        0.0085
hard        0.0085
level       0.0084
never       0.0083
stupid      0.0083
see         0.0083
pay         0.0082
terrible    0.0081
give        0.0080
playing     0.0077
love        0.0077
horrible    0.0076
boring      0.0075
people      0.0075

In [4]:
import re
from nltk.stem import PorterStemmer

ps = PorterStemmer()

LEXICON = {
    "quang_cao":      ["ad", "ads", "advertisement", "commercial", "popup", "banner"],
    "loi_hieu_nang":  ["crash", "bug", "freeze", "lag", "slow", "glitch", "stuck", "error",
                       "loading", "hang", "fix", "problem", "work", "open", "close", "force"],
    "cap_nhat":       ["update", "version", "upgrade", "latest"],
    "tien_thanh_toan":["money", "pay", "paid", "price", "expensive", "purchase", "charge",
                       "subscription", "refund", "scam", "cost", "buy"],
    "noi_dung_do_kho":["boring", "hard", "level", "difficult", "repetitive", "same"],
    "tai_khoan":      ["account", "login", "password", "sign", "verify", "register", "email"],
    "thiet_bi":       ["phone", "screen", "battery", "device", "storage", "memory", "space",
                       "install", "download"],
}
STEMS = {t: {ps.stem(w) for w in ws} for t, ws in LEXICON.items()}

def assign_topic(text):
    toks = [ps.stem(w) for w in re.findall(r"[a-z]{2,}", str(text).lower())]
    counts = {t: sum(tok in st for tok in toks) for t, st in STEMS.items()}
    best = max(counts.values())
    if best == 0:
        return "khac"
    return next(t for t, c in counts.items() if c == best)   # hoà thì lấy chủ đề đứng trước

df["topic"] = df["review_text"].map(assign_topic)
print(df["topic"].value_counts())
print((df["topic"].value_counts(normalize=True) * 100).round(1))

topic
loi_hieu_nang      1808
khac               1807
tien_thanh_toan     585
quang_cao           492
thiet_bi            385
noi_dung_do_kho     381
cap_nhat            351
tai_khoan           248
Name: count, dtype: int64
topic
loi_hieu_nang      29.8
khac               29.8
tien_thanh_toan     9.7
quang_cao           8.1
thiet_bi            6.4
noi_dung_do_kho     6.3
cap_nhat            5.8
tai_khoan           4.1
Name: proportion, dtype: float64


In [5]:
from collections import Counter

# 1) Từ nào đang kéo review vào "loi_hieu_nang"?
sub = df.loc[df["topic"] == "loi_hieu_nang", "review_text"]
cnt = Counter()
for t in sub:
    toks = {ps.stem(w) for w in re.findall(r"[a-z]{2,}", str(t).lower())}
    for s in STEMS["loi_hieu_nang"] & toks:
        cnt[s] += 1
print(cnt.most_common())

# 2) Nhóm "khac" còn chứa từ khoá nổi bật nào?
k = df.loc[df["topic"] == "khac", "review_text"].fillna("")
vk = TfidfVectorizer(lowercase=True, stop_words=sw, token_pattern=r"\b[a-zA-Z]{3,}\b",
                     min_df=5, max_df=0.5)
Xk = vk.fit_transform(k)
sk = pd.Series(np.asarray(Xk.mean(axis=0)).ravel(),
               index=vk.get_feature_names_out()).sort_values(ascending=False)
print(sk.head(50).round(4).to_string())

[('work', 543), ('fix', 459), ('load', 243), ('problem', 227), ('open', 209), ('slow', 205), ('crash', 180), ('error', 125), ('bug', 117), ('freez', 101), ('close', 97), ('forc', 80), ('glitch', 77), ('lag', 66), ('stuck', 49), ('hang', 25)]
game        0.0786
bad         0.0491
like        0.0285
hate        0.0281
time        0.0263
worst       0.0237
app         0.0197
fun         0.0196
waste       0.0196
play        0.0193
useless     0.0190
get         0.0190
sucks       0.0187
even        0.0146
ever        0.0143
really      0.0142
good        0.0134
much        0.0133
stupid      0.0120
want        0.0113
horrible    0.0113
annoying    0.0108
fake        0.0108
terrible    0.0108
see         0.0106
would       0.0105
please      0.0100
add         0.0099
need        0.0095
long        0.0092
little      0.0092
people      0.0092
playing     0.0092
every       0.0089
think       0.0088
make        0.0087
way         0.0087
wrong       0.0084
give        0.0082
stars       0.008

In [6]:
NEG_FAIL = re.compile(
    r"\b(not|no longer|never|stopped|stop|(?:does|do|did|is|was)n['’]?t|cannot|can['’]?t|won['’]?t)\b"
    r"[^.!?]{0,15}\b(work|open|load)", re.I)

LEXICON = {
    "quang_cao":        ["ad", "ads", "advertisement", "commercial", "popup", "banner"],
    "tien_thanh_toan":  ["money", "pay", "paid", "price", "expensive", "purchase", "charge",
                         "subscription", "refund", "scam", "cost", "buy"],
    "tai_khoan":        ["account", "login", "password", "sign", "verify", "register", "email"],
    "cap_nhat":         ["update", "version", "upgrade", "latest"],
    "thiet_bi":         ["phone", "screen", "battery", "device", "storage", "memory", "space",
                         "install", "download"],
    "noi_dung_do_kho":  ["boring", "hard", "level", "difficult", "repetitive", "same"],
    "de_xuat_tinh_nang":["add", "feature", "option", "wish", "improve", "suggest", "request"],
    "loi_hieu_nang":    ["crash", "bug", "freeze", "lag", "slow", "glitch", "stuck", "error",
                         "loading", "hang", "force"],
}
STEMS = {t: {ps.stem(w) for w in ws} for t, ws in LEXICON.items()}

def assign_topic(text):
    s = str(text)
    toks = [ps.stem(w) for w in re.findall(r"[a-z]{2,}", s.lower())]
    counts = {t: sum(tok in st for tok in toks) for t, st in STEMS.items()}
    if NEG_FAIL.search(s):
        counts["loi_hieu_nang"] += 1
    best = max(counts.values())
    if best == 0:
        return "khac"
    return next(t for t, c in counts.items() if c == best)   # hoà: chủ đề cụ thể đứng trước

df["topic"] = df["review_text"].map(assign_topic)
print(df["topic"].value_counts())
print((df["topic"].value_counts(normalize=True) * 100).round(1))

topic
khac                 1909
tien_thanh_toan       739
loi_hieu_nang         643
thiet_bi              633
cap_nhat              541
quang_cao             498
noi_dung_do_kho       420
tai_khoan             360
de_xuat_tinh_nang     314
Name: count, dtype: int64
topic
khac                 31.5
tien_thanh_toan      12.2
loi_hieu_nang        10.6
thiet_bi             10.5
cap_nhat              8.9
quang_cao             8.2
noi_dung_do_kho       6.9
tai_khoan             5.9
de_xuat_tinh_nang     5.2
Name: proportion, dtype: float64


In [7]:
df = df.sort_values("review_id").reset_index(drop=True)   # cố định thứ tự để cùng seed ra cùng mẫu

parts = []
for t in LEXICON:                                          # 8 chủ đề
    g = df[df["topic"] == t]
    parts.append(g.sample(n=12, random_state=RANDOM_STATE))
parts.append(df[df["topic"] == "khac"].sample(n=24, random_state=RANDOM_STATE))

sample = pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
sample.insert(0, "sample_id", range(1, len(sample) + 1))
assert len(sample) == 120

lab_dir = OUT_DIR / "labeling"
lab_dir.mkdir(parents=True, exist_ok=True)

# File được commit: chỉ có id, không có văn bản, không có nhãn máy
sample[["sample_id", "review_id"]].to_csv(lab_dir / "sample_ids.csv", index=False)

# File gán nhãn của bạn: có văn bản, nằm trong data/processed nên KHÔNG bị commit
blind = sample[["sample_id", "review_id", "review_text"]].copy()
blind["topic_label"] = ""
blind.to_csv(PROC_DIR / "labeling_D_blind.csv", index=False, encoding="utf-8-sig")

print(sample["topic"].value_counts())

topic
khac                 24
cap_nhat             12
quang_cao            12
thiet_bi             12
tai_khoan            12
noi_dung_do_kho      12
de_xuat_tinh_nang    12
tien_thanh_toan      12
loi_hieu_nang        12
Name: count, dtype: int64
